In [ ]:
!pip install psycopg2-binary -q

# Connect to the database

In [ ]:
import sqlalchemy
import pandas as pd
import numpy as np
from google.colab import userdata

In [ ]:
DB_USER = userdata.get('DB_USER')
DB_PASSWORD = userdata.get('DB_PASSWORD')
DB_HOST = userdata.get('DB_HOST')
DB_NAME = userdata.get('DB_NAME')
DB_PORT = "5432"

In [ ]:
connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = sqlalchemy.create_engine(connection_string)

# Analysis

In [ ]:
query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public';
"""
df = pd.read_sql(query, engine)
print(df)

In [ ]:
query = "SELECT * FROM smart_device_readings LIMIT 5;"
df = pd.read_sql(query, engine)

In [ ]:
df

In [ ]:
test_reverse_query = """
SELECT * FROM smart_device_readings
WHERE gateway_serial = 'EHM54090515'
ORDER BY timestamp DESC
LIMIT 5;
"""
test_df_reverse = pd.read_sql(test_reverse_query, engine)

In [ ]:
test_df_reverse

In [ ]:
test_df_reverse.iloc[:, :20]

In [ ]:
columns_abmf = test_df_reverse.columns

In [ ]:
non_null_columns = test_df_reverse.dropna(axis=1).columns

In [ ]:
test_df_reverse['import_active_energy_overall_total'].dropna()

In [ ]:
with open('my_output.txt', 'w') as f:
    f.write('\n'.join(test_df_reverse))

In [ ]:
'\n'.join(test_df_reverse)

In [ ]:
empty_columns = test_df_reverse.columns[test_df_reverse.isna().all()].tolist()

In [ ]:
with open('null_columns.txt', 'w') as f:
    f.write('\n'.join(empty_columns))

In [ ]:
test_df_reverse[empty_columns]

In [ ]:
# Testing data for 16 and 17
data_16_17_may = """
SELECT * FROM smart_device_readings
WHERE gateway_serial = 'EHM54090515' AND date BETWEEN '2026-05-16' AND '2026-05-17'
ORDER BY timestamp DESC;
"""
df_16_17_may = pd.read_sql(data_16_17_may, engine)

In [ ]:
df_16_17_may.describe()

In [ ]:
import datetime
df_16_17_may[df_16_17_may['date'] == datetime.date(2026, 5, 16)]

In [ ]:
# Reorder timestamp
df_16_17_may = df_16_17_may.sort_values(by='timestamp', ascending=True).reset_index(drop=True)

In [ ]:
df_16_17_may[df_16_17_may['date'] == datetime.date(2026, 5, 17)][61:121]

In [ ]:
df_16_17_may[df_16_17_may['date'] == datetime.date(2026, 5, 17)][120:180]

In [ ]:
df_16_17_may['date'][0]

It is not certain how power outages and conditions where there is no grid supply or generator active

In [ ]:
df_16_17_may.dropna(axis = 1)

In [ ]:
empty_columns = df_16_17_may.columns[df_16_17_may.isna().all()].tolist()

In [ ]:
with open('null_columns.txt', 'w') as f:
    f.write('\n'.join(empty_columns))

In [ ]:
df_16_17_may.iloc[:, 40:55]

## 19/05 Exploration

In [ ]:
# After fix data

all_data_after_fix = """
SELECT * FROM public.smart_device_readings
WHERE gateway_serial = 'EHM54090515' AND timestamp > '2026-05-15 12:00:00'
ORDER BY timestamp DESC;
"""
all_data_after_fix_df = pd.read_sql(all_data_after_fix, engine)

Time ran: 19/05/2026 11:32 pm

In [ ]:
all_data_after_fix_df.head().describe().iloc[:, 40:60]

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
all_data_after_fix_df.head()

In [ ]:
sum(all_data_after_fix_df['input_channel_1_current_demand'] == all_data_after_fix_df['power_asset_1_current_phase_a'])

In [ ]:
all_data_after_fix_df[['power_asset_1_current_phase_a', 'power_asset_1_current_phase_b', 'power_asset_1_current_phase_c', 'input_channel_1_current_demand']]

In [ ]:
useful_df = all_data_after_fix_df[['timestamp', 'dominant_asset',	'active_assets', 'line_to_neutral_voltage_phase_a', 'line_to_neutral_voltage_phase_b', 'line_to_neutral_voltage_phase_c', 'line_current_overall_phase_a', 'line_current_overall_phase_b', 'line_current_overall_phase_c', 'line_current_overall_neutral', 'active_power_overall_total','power_factor_overall_phase_a', 'power_factor_overall_phase_b', 'power_factor_overall_phase_c', 'active_power_overall_phase_a', 'active_power_overall_phase_b', 'active_power_overall_phase_c', 'voltage_unbalance_factor', 'current_unbalance_factor', 'frequency', 'power_factor_overall', 'reactive_power_overall_total', 'reactive_power_overall_phase_a', 'reactive_power_overall_phase_b', 'reactive_power_overall_phase_c', 'apparent_power_overall_total', 'reactive_power_overall', 'total_harmonic_distortion_current_phase_a', 'total_harmonic_distortion_current_phase_b', 'total_harmonic_distortion_current_phase_c', 'load', 'energy', 'Distributed_Electricity', 'transformer_capacity', 'transformer_load_percentage', 'power_asset_1_kW', 'power_asset_1_kvar', 'power_asset_1_kVA', 'power_asset_1_kWh', 'power_asset_1_kvarh', 'power_asset_1_hours', 'power_asset_1_current_phase_a', 'power_asset_1_current_phase_b', 'power_asset_1_current_phase_c', 'power_asset_1_neutral_current', 'power_asset_2_kW', 'power_asset_2_kvar', 'power_asset_2_kVA', 'power_asset_2_kWh', 'power_asset_2_kvarh', 'power_asset_2_hours', 'power_asset_2_current_phase_a', 'power_asset_2_current_phase_b', 'power_asset_2_current_phase_c', 'power_asset_2_neutral_current']].copy()

In [ ]:
useful_df.describe()

### ABA Data

In [ ]:
# For aba

aba_data_10_16_may = """
SELECT * FROM public.smart_device_readings
WHERE gateway_serial = 'EHM21120502' AND timestamp BETWEEN '2026-05-10 00:00:00' AND '2026-05-16 23:59:59'
ORDER BY timestamp DESC;
"""
aba_data_10_16_may_df = pd.read_sql(aba_data_10_16_may, engine)

In [ ]:
aba_data_10_16_may_df.head()

In [ ]:
# Checking to see if it's similar to the important stuffs from Head Office
useful_data_aba = aba_data_10_16_may_df[['timestamp', 'dominant_asset',	'active_assets', 'line_to_neutral_voltage_phase_a', 'line_to_neutral_voltage_phase_b', 'line_to_neutral_voltage_phase_c', 'line_current_overall_phase_a', 'line_current_overall_phase_b', 'line_current_overall_phase_c', 'line_current_overall_neutral', 'active_power_overall_total','power_factor_overall_phase_a', 'power_factor_overall_phase_b', 'power_factor_overall_phase_c', 'power_factor_overall', 'voltage_unbalance_factor', 'current_unbalance_factor', 'frequency', 'reactive_power_overall_total', 'apparent_power_overall_total', 'reactive_power_overall', 'total_harmonic_distortion_current_phase_a', 'total_harmonic_distortion_current_phase_b', 'total_harmonic_distortion_current_phase_c', 'load', 'energy', 'Distributed_Electricity', 'transformer_capacity', 'transformer_load_percentage', 'power_asset_1_kW', 'power_asset_1_kvar', 'power_asset_1_kVA', 'power_asset_1_kWh', 'power_asset_1_kvarh', 'power_asset_1_hours', 'power_asset_1_current_phase_a', 'power_asset_1_current_phase_b', 'power_asset_1_current_phase_c', 'power_asset_1_neutral_current', 'power_asset_2_kW', 'power_asset_2_kvar', 'power_asset_2_kVA', 'power_asset_2_kWh', 'power_asset_2_kvarh', 'power_asset_2_hours', 'power_asset_2_current_phase_a', 'power_asset_2_current_phase_b', 'power_asset_2_current_phase_c', 'power_asset_2_neutral_current']].copy()

In [ ]:
useful_data_aba.describe()

In [ ]:
useful_data_aba.active_assets.value_counts()

In [ ]:
useful_data_aba.dominant_asset.value_counts()

In [ ]:
aba_data_10_16_may_df[aba_data_10_16_may_df['active_assets'] == 'Grid,Generator 1']

In [ ]:
aba_data_10_16_may_df[aba_data_10_16_may_df['active_assets'] == 'Grid,Generator 1'].describe()

In [ ]:
useful_data_aba.head()

In [ ]:
useful_data_aba[useful_data_aba[]]

In [ ]:
# Raymond said some issues have been fixed.

most_recent_data_19_05_26 = """
SELECT * FROM public.smart_device_readings
ORDER BY timestamp DESC
LIMIT 100;
"""
most_recent_data_19_05_26_df = pd.read_sql(most_recent_data_19_05_26, engine)

In [ ]:
most_recent_data_19_05_26_df.tail()

In [ ]:
# Raymond said some issues have been fixed.

most_recent_data_19_05_26_asc = """
SELECT * FROM public.smart_device_readings
WHERE gateway_serial = 'EHM54090515'
ORDER BY timestamp ASC
LIMIT 100;
"""
most_recent_data_19_05_26_asc_df = pd.read_sql(most_recent_data_19_05_26_asc, engine)

In [ ]:
most_recent_data_19_05_26_asc_df.head()

In [ ]:
most_recent_data_19_05_26_df['active_assets'].value_counts()

In [ ]:
most_recent_data_19_05_26_df[['input_channel_1_current', 'line_current_overall_phase_a', 'power_asset_1_current_phase_a', 'input_channel_4_current', 'power_asset_2_current_phase_a']]

21/05

In [ ]:
# Trying to troubleshoot why line_current_overall_phase_a isn't the same with the remaining two columns

most_recent_data_21_05_26_hoffice = """
SELECT * FROM public.smart_device_readings
WHERE gateway_serial = 'EHM54090515'
ORDER BY timestamp DESC
LIMIT 100;
"""
most_recent_data_21_05_26_hoffice_df = pd.read_sql(most_recent_data_21_05_26_hoffice, engine)

In [ ]:
most_recent_data_21_05_26_hoffice_df[['line_current_overall_phase_a','input_channel_4_current', 'power_asset_2_current_phase_a']]

In [ ]:
(most_recent_data_21_05_26_hoffice_df['line_current_overall_phase_a'] > most_recent_data_21_05_26_hoffice_df['input_channel_4_current']).value_counts()

site_information

core_devicetariff

companies

power_source_information

asset_classification

In [ ]:
# Let's try to join some tables
most_recent_core_devicetariff = """
SELECT * FROM public.core_devicetariff
LIMIT 100;
"""
most_recent_core_devicetariff_df = pd.read_sql(most_recent_core_devicetariff, engine)

In [ ]:
most_recent_core_devicetariff_df

In [ ]:
most_recent_site_information = """
SELECT * FROM public.site_information
LIMIT 100;
"""
most_recent_site_information_df = pd.read_sql(most_recent_site_information, engine)

In [ ]:
most_recent_site_information_df

In [ ]:
most_recent_companies = """
SELECT * FROM public.companies
LIMIT 100;
"""
most_recent_companies_df = pd.read_sql(most_recent_companies, engine)

In [ ]:
most_recent_companies_df

In [ ]:
most_recent_power_source_information = """
SELECT * FROM public.power_source_information
LIMIT 100;
"""
most_recent_power_source_information_df = pd.read_sql(most_recent_power_source_information, engine)

In [ ]:
most_recent_power_source_information_df

In [ ]:
most_recent_asset_classification = """
SELECT * FROM public.site_information
LIMIT 100;
"""
most_recent_site_information_df = pd.read_sql(most_recent_site_information, engine)

24/05

In [ ]:
# Finalizing on which columns are useful

most_recent_data_24_05_26_hoffice = """
SELECT * FROM public.smart_device_readings
WHERE gateway_serial = 'EHM54090515'
ORDER BY timestamp DESC
LIMIT 100;
"""
most_recent_data_24_05_26_hoffice_df = pd.read_sql(most_recent_data_24_05_26_hoffice, engine)

In [ ]:
most_recent_data_24_05_26_hoffice_df.head()

In [ ]:
most_recent_data_24_05_26_hoffice_df.dropna(axis=1, how='all')

In [ ]:
most_recent_data_24_05_26_hoffice_df.dropna(axis=1)

In [ ]:
most_recent_data_24_05_26_hoffice_df.dropna(axis=1).loc[:, (most_recent_data_24_05_26_hoffice_df != 0).any(axis=0)]

In [ ]:
# removing the columns with 0 values allthrough
useful_columns = most_recent_data_24_05_26_hoffice_df.dropna(axis=1).loc[:, (most_recent_data_24_05_26_hoffice_df != 0).any(axis=0)].columns.tolist()

In [ ]:
most_recent_data_24_05_26_hoffice_df.dropna(axis=1).columns.tolist()

In [ ]:
with open('useful_columns.txt', 'w') as f:
    f.write('\n'.join(useful_columns))

In [ ]:
# Trying to see the other columns

remaining_columns_df = most_recent_data_24_05_26_hoffice_df.dropna(axis=1).drop(columns=useful_columns).copy()

In [ ]:
remaining_columns_df.sample().head()

25/05

In [ ]:
# Finalizing on which columns are useful

abmf_hoffice_20_24_data = """
SELECT * FROM public.smart_device_readings
WHERE gateway_serial = 'EHM54090515' AND timestamp BETWEEN '2026-05-20 00:00:00' AND '2026-05-24 23:59:59'
ORDER BY timestamp DESC;
"""

abmf_hoffice_20_24_df = pd.read_sql(abmf_hoffice_20_24_data, engine)

In [ ]:
# removing the columns with 0 values allthrough
useful_columns = abmf_hoffice_20_24_df.dropna(axis=1).loc[:, (abmf_hoffice_20_24_df != 0).any(axis=0)].columns.tolist()

In [ ]:
len(useful_columns)

In [ ]:
with open('useful_columns.txt', 'w') as f:
    f.write('\n'.join(useful_columns))

In [ ]:
abmf_hoffice_20_24_df.dropna(axis=1).drop(columns=useful_columns).columns.tolist()

In [ ]:
# This removes the limits ONLY for the code inside the block
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(abmf_hoffice_20_24_df.sample().dropna(how='all', axis=1))

In [ ]:
(round(abmf_hoffice_20_24_df['active_power_overall_total'], 2) == round(abmf_hoffice_20_24_df['active_power_overall_phase_a'] + abmf_hoffice_20_24_df['active_power_overall_phase_b'] + abmf_hoffice_20_24_df['active_power_overall_phase_c'], 2)).value_counts()

In [ ]:
abmf_hoffice_20_24_df['frequency'].hist()

In [ ]:
(abmf_hoffice_20_24_df['reactive_power_overall'] == abmf_hoffice_20_24_df['reactive_power_overall_total']).value_counts()

In [ ]:
abmf_hoffice_20_24_df[abmf_hoffice_20_24_df['reactive_power_overall'] != abmf_hoffice_20_24_df['reactive_power_overall_total']][['reactive_power_overall', 'reactive_power_overall_total']]

In [ ]:
(abmf_hoffice_20_24_df['input_channel_1_active_power'] == abmf_hoffice_20_24_df['user_channel_1_active_power']).value_counts()

In [ ]:
abmf_hoffice_20_24_df[['input_channel_7_current_demand', 'input_channel_8_current_demand', 'input_channel_9_current_demand']].sum(axis=1)

In [ ]:
abmf_hoffice_20_24_df['input_channel_4_power_factor'] == abmf_hoffice_20_24_df['user_channel_4_power_factor']

In [ ]:
abmf_hoffice_20_24_df['average_voltage_unbalance_factor'] == abmf_hoffice_20_24_df['voltage_unbalance_factor']

In [ ]:
abmf_hoffice_20_24_df[['load', 'energy']]

In [ ]:
abmf_hoffice_20_24_df['updated_power_factor_overall'] = abmf_hoffice_20_24_df['active_power_overall_total'] / abmf_hoffice_20_24_df['apparent_power_overall_total']

In [ ]:
useful_column_data = abmf_hoffice_20_24_df.dropna(axis=1).loc[:, (abmf_hoffice_20_24_df != 0).any(axis=0)].copy()

In [ ]:
better_data = useful_column_data[useful_column_data['load'] > 0].copy()

In [ ]:
(better_data['load'] == better_data['power_asset_1_kW']).value_counts()

In [ ]:
(better_data['load'] == better_data['power_asset_2_kW']).value_counts()

In [ ]:
better_data[['load', 'power_asset_1_kW', 'power_asset_2_kW']].sample(15)

In [ ]:
better_data['power_asset_1_kWh']

In [ ]:
better_data[['power_asset_1_kWh', 'power_asset_2_kWh', 'dominant_asset_kWh', 'total_system_hours', 'total_system_kWh']]

In [ ]:
better_data['Distributed_Electricity']/better_data['load']

In [ ]:
better_data[better_data['dominant_asset_kWh'] != better_data['total_system_kWh']][['dominant_asset_kWh', 'total_system_kWh', 'power_asset_2_kWh']]

In [ ]:
better_data['active_assets'].value_counts()

In [ ]:
(better_data['dominant_asset_kWh'] != better_data['total_system_kWh']).value_counts()

In [ ]:
abmf_hoffice_20_24_df['active_assets'].value_counts()

In [ ]:
len(abmf_hoffice_20_24_df)

In [ ]:
abmf_hoffice_20_24_df.head()

In [ ]:
better_data['workhour'].value_counts()

In [ ]:
abmf_hoffice_20_24_df['transformer_capacity']

In [ ]:
phase_cols = ['line_to_neutral_voltage_phase_a', 'line_to_neutral_voltage_phase_b', 'line_to_neutral_voltage_phase_c']

better_data["I_avg"] = better_data[phase_cols].mean(axis=1)

# 2. Find the maximum absolute deviation from the average
# .sub(..., axis=0) subtracts the average row-wise
max_deviation = better_data[phase_cols].sub(better_data["I_avg"], axis=0).abs().max(axis=1)

# 3. Calculate the unbalance percentage
# We use np.where to safely handle cases where average current is 0 (to avoid division by zero)
better_data["Current_Unbalance_Pct"] = np.where(
    better_data["I_avg"] > 0, (max_deviation / better_data["I_avg"]) * 100, 0.0
)

# Optional: Drop the temporary average column if you don't need it
better_data = better_data.drop(columns=["I_avg"])

In [ ]:
better_data[['Current_Unbalance_Pct', 'average_voltage_unbalance_factor', 'voltage_unbalance_factor']]

In [ ]:
better_data['calculated_power_factor_channel_1'] = better_data['input_channel_1_active_power'] / better_data['input_channel_1_apparent_power']

In [ ]:
better_data[['input_channel_1_current', 'calculated_power_factor_channel_1', 'input_channel_1_power_factor']]

26/05

In [ ]:
# Finalizing on which columns are useful

abmf_hoffice_20_25_data = """
SELECT
    timestamp,
    gateway_serial,
    active_power_overall_total,
    apparent_power_overall_total,
    power_factor_overall,
    "total_system_kWh",
    load,
    frequency,
    active_assets,
    active_asset_count,
    workhour,
    transformer_capacity,
    transformer_load_percentage,
    line_to_neutral_voltage_phase_a,
    line_to_neutral_voltage_phase_b,
    line_to_neutral_voltage_phase_c,
    line_current_overall_phase_a,
    line_current_overall_phase_b,
    line_current_overall_phase_c,
    power_factor_overall_phase_a,
    power_factor_overall_phase_b,
    power_factor_overall_phase_c,
    voltage_unbalance_factor,
    current_unbalance_factor,
    total_harmonic_distortion_current_phase_a,
    total_harmonic_distortion_current_phase_b,
    total_harmonic_distortion_current_phase_c
FROM public.smart_device_readings
WHERE gateway_serial = 'EHM54090515' AND timestamp BETWEEN '2026-05-20 00:00:00' AND '2026-05-25 23:59:59'
ORDER BY timestamp DESC;
"""

abmf_hoffice_20_25_df = pd.read_sql(abmf_hoffice_20_25_data, engine)

In [ ]:
abmf_hoffice_20_25_df.head()

In [ ]:
with pd.option_context('display.max_columns', None):
    display(abmf_hoffice_20_25_df[abmf_hoffice_20_25_df['active_assets'] != 'Grid'].head())

I made the following corrections:
1. Workhour
2. Power Factor calculation issue especially when there's no current in other some phases
3. Updating transformer_capacity into something that will account for the particular power asset in use not just the transformer especially when used in load_rate calculations
4. The calculation we're using for load_rate is (active_power/power_factor) / capacity, right? Can we use apparent_power/capacity in the case the power_factor_overall from the meter is inaccurate? It's like this is what is in your data_mapping document
5. Workhour is updated.

In [ ]:
abmf_hoffice_20_25_df['updated_power_factor_overall'] = abmf_hoffice_20_25_df['active_power_overall_total'] / abmf_hoffice_20_25_df['apparent_power_overall_total']
abmf_hoffice_20_25_df['updated_load_rate'] = abmf_hoffice_20_25_df['apparent_power_overall_total'] / abmf_hoffice_20_25_df['updated_power_asset_capacity']

In [ ]:
conditions = [
    abmf_hoffice_20_25_df['active_assets'] == 'Grid',
    abmf_hoffice_20_25_df['active_assets'] == 'Generator 1'
]

choices = [200, 88]

abmf_hoffice_20_25_df['updated_power_asset_capacity'] = np.select(conditions, choices, default=0)

In [ ]:
# Let me just see everything at once
import pandas as pd

# Remove the limit on the number of columns displayed
pd.set_option('display.max_columns', None)

In [ ]:
abmf_hoffice_20_25_df.sample(5)

In [ ]:
abmf_hoffice_20_25_df['active_assets'].value_counts()

In [ ]:
len(abmf_hoffice_20_25_df)

In [ ]:
abmf_hoffice_20_25_df[abmf_hoffice_20_25_df['active_assets'] == ""].head()

In [ ]:
abmf_hoffice_20_25_df.drop(['transformer_capacity', 'transformer_load_percentage', 'power_factor_overall'], axis=1, inplace=True)

In [ ]:
abmf_hoffice_20_25_df.sample(5)

In [ ]:
import pandas as pd

# 1. Ensure the timestamp column is recognized as a datetime object
abmf_hoffice_20_25_df['timestamp'] = pd.to_datetime(abmf_hoffice_20_25_df['timestamp'])

# 1b. Filter out weekends (0=Monday, 4=Friday, 5=Saturday, 6=Sunday)
# This keeps only Monday through Friday
weekday_df = abmf_hoffice_20_25_df[abmf_hoffice_20_25_df['timestamp'].dt.dayofweek < 5].copy()

# 2. Extract the hour (0-23) into a temporary column on the filtered data
weekday_df['hour_of_day'] = weekday_df['timestamp'].dt.hour

# 3. Create a cross-tabulation table to see the distribution
hourly_profile = pd.crosstab(
    index=weekday_df['hour_of_day'],
    columns=weekday_df['workhour'],
    normalize='index'  # Shows percentages instead of raw counts
) * 100

print(hourly_profile)

In [ ]:
# 1. Ensure timestamp is datetime
abmf_hoffice_20_25_df['timestamp'] = pd.to_datetime(abmf_hoffice_20_25_df['timestamp'])

# 2. Build the filters
is_weekday = abmf_hoffice_20_25_df['timestamp'].dt.dayofweek < 5
is_target_hour = abmf_hoffice_20_25_df['timestamp'].dt.hour.isin([15, 16])

# 3. Apply filters
investigation_df = abmf_hoffice_20_25_df[is_weekday & is_target_hour].copy()

# 4. Add human-readable day/time columns to make anomalies easy to spot
investigation_df['extracted_date'] = investigation_df['timestamp'].dt.date
investigation_df['extracted_time'] = investigation_df['timestamp'].dt.time
investigation_df['day_name'] = investigation_df['timestamp'].dt.day_name()

# 5. Pull a random sample of 15 rows
# (Change random_state or remove it to see a different batch of data)
sample_data = investigation_df.sample(n=15, random_state=42)

# Display specific columns first to make spotting the issue easier
debug_cols = ['timestamp', 'day_name', 'workhour', 'gateway_serial']
sample_data[debug_cols + [col for col in sample_data.columns if col not in debug_cols]]

In [ ]:
# See which exact dates have a '0' during these hours
anomaly_by_date = investigation_df[investigation_df['workhour'] == 0].groupby('extracted_date').size()
print("Dates where workhour dropped to 0 during 3 PM/4 PM:")
print(anomaly_by_date.head(20))

# See if it's breaking at a specific minute (e.g., exactly at 15:30:00)
investigation_df['extracted_minute'] = investigation_df['timestamp'].dt.minute
anomaly_by_minute = pd.crosstab(investigation_df['extracted_minute'], investigation_df['workhour'])
print("\nDistribution by minute look:")
print(anomaly_by_minute.head(10))

In [ ]:
# 1. Ensure the timestamp column is recognized as a datetime object
abmf_hoffice_20_25_df['timestamp'] = pd.to_datetime(abmf_hoffice_20_25_df['timestamp'])

# 2. Filter out weekends AND Wednesdays
# (0=Mon, 1=Tue, 2=Wed, 3=Thu, 4=Fri, 5=Sat, 6=Sun)
no_wed_df = abmf_hoffice_20_25_df[
    (abmf_hoffice_20_25_df['timestamp'].dt.dayofweek < 5) &
    (abmf_hoffice_20_25_df['timestamp'].dt.dayofweek != 2)
].copy()

# 3. Extract the hour (0-23) into a temporary column
no_wed_df['hour_of_day'] = no_wed_df['timestamp'].dt.hour

# 4. Create the cross-tabulation table to see the new distribution
hourly_profile = pd.crosstab(
    index=no_wed_df['hour_of_day'],
    columns=no_wed_df['workhour'],
    normalize='index'  # Shows percentages instead of raw counts
) * 100

print("Hourly profile (Mon, Tue, Thu, Fri only):")
print(hourly_profile)

In [ ]:
import pandas as pd

# 1. Ensure the timestamp column is recognized as a datetime object
abmf_hoffice_20_25_df['timestamp'] = pd.to_datetime(abmf_hoffice_20_25_df['timestamp'])

# 2. Filter out weekends AND exclude May 25, 2026
# (Wednesdays are now safely left in the data)
filtered_df = abmf_hoffice_20_25_df[
    (abmf_hoffice_20_25_df['timestamp'].dt.dayofweek < 5) &
    (abmf_hoffice_20_25_df['timestamp'].dt.date != pd.to_datetime('2026-05-25').date())
].copy()

# 3. Extract the hour (0-23) into a temporary column
filtered_df['hour_of_day'] = filtered_df['timestamp'].dt.hour

# 4. Create the cross-tabulation table to see the distribution
hourly_profile = pd.crosstab(
    index=filtered_df['hour_of_day'],
    columns=filtered_df['workhour'],
    normalize='index'  # Shows percentages instead of raw counts
) * 100

print("Hourly profile (Weekdays, excluding 25/05/2026):")
print(hourly_profile)

In [ ]:
with pd.option_context('display.max_rows', None):
    display(abmf_hoffice_20_25_df[abmf_hoffice_20_25_df['timestamp'].dt.date == pd.to_datetime('2026-05-25').date()][['timestamp', 'workhour']])

The issue was just for Wednesday (25th). Confirm with other days else the workhour column will just be from 8 am - 4 pm.

### Filling of data to create the full picture

In [ ]:
abmf_hoffice_20_25_df_copy = abmf_hoffice_20_25_df.copy()

In [ ]:
# 1. Convert the column to datetime (just in case it loaded as a string/object)
abmf_hoffice_20_25_df_copy["timestamp"] = pd.to_datetime(abmf_hoffice_20_25_df_copy["timestamp"])

# 2. Set the column as the index
abmf_hoffice_20_25_df_copy = abmf_hoffice_20_25_df_copy.set_index("timestamp")

# 3. Crucial: Sort the index chronologically
abmf_hoffice_20_25_df_copy = abmf_hoffice_20_25_df_copy.sort_index()

In [ ]:
df_filled = abmf_hoffice_20_25_df_copy.resample("1min").first()

In [ ]:
df_filled.isna().sum()

In [ ]:
df_filled.head()

In [ ]:
len(df_filled)

In [ ]:
df_filled.head(50)

In [ ]:
filled_index_values = df_filled.index.tolist()

In [ ]:
with open('filled_index_values.txt', 'w') as f:
    for item in filled_index_values:
        f.write(str(item) + '\n')

In [ ]:
# --- 3. Sessionization Logic ---
is_missing = df_filled["gateway_serial"].isna()
df_filled["session_marker"] = (is_missing != is_missing.shift()).cumsum()

outages_df = df_filled[is_missing].reset_index()
outages_df = outages_df.rename(columns={"index": "timestamp"})


# --- 4. Calculate Outage Durations ---
outage_analysis = (
    outages_df.groupby("session_marker")
    .agg(
        outage_start=("timestamp", "min"),
        outage_end=("timestamp", "max"),
        duration_minutes=("timestamp", "count"),
    )
    .reset_index(drop=True)
)


# --- 5. Asset Tracking: Before & After Lookups ---
# Look back 1 minute from start
pre_outage_timestamps = outage_analysis["outage_start"] - pd.Timedelta(minutes=1)
outage_analysis["asset_before_outage"] = pre_outage_timestamps.map(
    abmf_hoffice_20_25_df["active_assets"]
)

# Look forward 1 minute from end
post_outage_timestamps = outage_analysis["outage_end"] + pd.Timedelta(minutes=1)
outage_analysis["asset_after_outage"] = post_outage_timestamps.map(
    abmf_hoffice_20_25_df["active_assets"]
)


print("--- Final Power Outage Sessionization Report ---")
print(outage_analysis)

In [ ]:
# --- 3. Sessionization Logic ---
is_missing = df_filled["gateway_serial"].isna()
df_filled["session_marker"] = (is_missing != is_missing.shift()).cumsum()

outages_df = df_filled[is_missing].reset_index()
outages_df = outages_df.rename(columns={"index": "timestamp"})


# --- 4. Calculate Outage Durations ---
outage_analysis = (
    outages_df.groupby("session_marker")
    .agg(
        outage_start=("timestamp", "min"),
        outage_end=("timestamp", "max"),
        duration_minutes=("timestamp", "count"),
    )
    .reset_index(drop=True)
)


# --- 5. Asset Tracking: Before & After Lookups ---
# Look back 1 minute from start
pre_outage_timestamps = outage_analysis["outage_start"] - pd.Timedelta(minutes=1)
outage_analysis["asset_before_outage"] = pre_outage_timestamps.map(
    df_filled["active_assets"]
)

# Look forward 1 minute from end
post_outage_timestamps = outage_analysis["outage_end"] + pd.Timedelta(minutes=1)
outage_analysis["asset_after_outage"] = post_outage_timestamps.map(
    df_filled["active_assets"]
)


print("--- Final Power Outage Sessionization Report ---")
print(outage_analysis)

In [ ]:
outage_analysis

In [ ]:
with pd.option_context('display.max_rows', None):
    display(outage_analysis)

No insights from this yet. Especially because of the active_assets that don't have load.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# 1. Reset the index to turn 'timestamp' into a regular column for Seaborn
df = df_filled.reset_index()

# Double check that the index column is named correctly
if "timestamp" not in df.columns and "index" in df.columns:
    df = df.rename(columns={"index": "timestamp"})

# 2. Clean, sort, and parse data
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

# Replace empty strings and NaNs with clear tracking labels
df["active_assets"] = df["active_assets"].replace(
    {"": "None / Standby", np.nan: "No Power"}
)

# 3. Create a unique segment ID every time the asset switches
df["segment_id"] = (df["active_assets"] != df["active_assets"].shift()).cumsum()

# 4. Calculate Peaks and Baseline Loads
peak_idx = df["load"].idxmax()
peak_load = df["load"].max()
peak_time = df.loc[peak_idx, "timestamp"]
baseline_load = df["load"].quantile(0.05)

# 5. Generate the Visualization
plt.figure(figsize=(15, 6))

# Map your specific assets to dedicated colors (added 'No Power')
custom_palette = {
    "Grid": "#1f77b4",  # Blue
    "Generator 1": "#ff7f0e",  # Orange
    "None / Standby": "#7f7f7f",  # Gray
    "No Power": "#000000",  # Black
}

# Plot the load lines
sns.lineplot(
    data=df,
    x="timestamp",
    y="load",
    hue="active_assets",
    units="segment_id",
    estimator=None,
    palette=custom_palette,
    linewidth=2,
    alpha=0.8,
)

# 6. Highlight Baseline and Peaks
plt.axhline(
    y=baseline_load,
    color="red",
    linestyle="--",
    linewidth=1.5,
    label=f"Baseline Load ({baseline_load:.2f})",
)

plt.scatter(
    peak_time, peak_load, color="darkred", s=120, zorder=5, label="Peak Load Location"
)
plt.annotate(
    f"PEAK LOAD: {peak_load:.2f}\n{peak_time.strftime('%Y-%m-%d %H:%M')}",
    xy=(peak_time, peak_load),
    xytext=(25, 10),
    textcoords="offset points",
    arrowprops=dict(arrowstyle="->", color="darkred", lw=1.5),
    fontweight="bold",
    color="darkred",
    bbox=dict(boxstyle="round,pad=0.3", fc="yellow", alpha=0.3, ec="darkred"),
)

# Formatting, Labels, and Legend cleanup
plt.title(
    "Facility Load Profile Over Time (Color-Coded by Energy Source)",
    fontsize=14,
    pad=15,
    weight="bold",
)
plt.xlabel("Timeline", fontsize=12)
plt.ylabel("Load Magnitude", fontsize=12)
plt.grid(True, linestyle=":", alpha=0.5)

# Deduplicate legend entries
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(
    by_label.values(), by_label.keys(), title="System Labels", loc="upper right"
)

plt.tight_layout()
plt.show()

In [ ]:
df_filled.columns

In [ ]:
df_filled[[]'load', 'total_system_kWh']